In [1]:
import time

# We will need Qibo to write our quantum circuit
from qibo import Circuit, gates, set_backend

# We will need mpstab to construct a surrogate
from mpstab import HSMPO

In [2]:
set_backend("qibojit", platform="numba")

[Qibo 0.2.23|INFO|2026-02-23 15:44:16]: Using qibojit (numba) backend on /CPU:0


In [4]:
n = 50

circ = Circuit(nqubits=n)
circ.add(gates.H(0))
[circ.add(gates.CNOT(q, q+1)) for q in range(n - 1)]

circ.draw()

0 :     ─H─o────────────────────────────────────────────────────────────────── ...
1 :     ───X─o──────────────────────────────────────────────────────────────── ...
2 :     ─────X─o────────────────────────────────────────────────────────────── ...
3 :     ───────X─o──────────────────────────────────────────────────────────── ...
4 :     ─────────X─o────────────────────────────────────────────────────────── ...
5 :     ───────────X─o──────────────────────────────────────────────────────── ...
6 :     ─────────────X─o────────────────────────────────────────────────────── ...
7 :     ───────────────X─o──────────────────────────────────────────────────── ...
8 :     ─────────────────X─o────────────────────────────────────────────────── ...
9 :     ───────────────────X─o──────────────────────────────────────────────── ...
10:     ─────────────────────X─o────────────────────────────────────────────── ...
11:     ───────────────────────X─o──────────────────────────────────────────── ...
12: 

In [5]:
surry = HSMPO(circ)

In [6]:
# %time
surry.expectation(
    observable="Z"*n
)

/Users/robbiati/Documents/tac/lib/python3.12/site-packages/cotengra/hyperoptimizers/hyper.py:36: UserWarning: Couldn't import `kahypar` - skipping from default hyper optimizer and using basic `labels` method instead. `kahypar` is highly recommended for the best quality contraction paths.
  warnings.warn(


np.float64(1.0)

In [7]:
from mpstab.models.ansatze import HardwareEfficient 
from mpstab.utils import obs_string_to_qibo_hamiltonian

def execute_circuit(nqubits, nlayers):
    obs_str = "Z" * nqubits

    ans = HardwareEfficient(nqubits=nqubits, nlayers=nlayers)
    hs = HSMPO(ansatz=ans)

    it = time.time()
    expval_mpstab = hs.expectation(obs_str)
    time_mpstab = time.time() - it
    
    ham = obs_string_to_qibo_hamiltonian(obs_str)
    it = time.time()
    expval_qibo = ham.expectation_from_state(
        ans.circuit().state()
    )
    time_qibo = time.time() - it
    return (expval_mpstab, expval_qibo), (time_mpstab, time_qibo)

In [8]:
(expval_mpstab, expval_qibo), (time_mpstab, time_qibo) = execute_circuit(nqubits=12, nlayers=2)

print(f"MPSTAB expectation value: {expval_mpstab}, time taken: {time_mpstab:.4f} seconds")
print(f"Qibo expectation value: {expval_qibo}, time taken: {time_qibo:.4f} seconds")

/Users/robbiati/Documents/tac/lib/python3.12/site-packages/autoray/autoray.py:96: RuntimeWarning: divide by zero encountered in matmul
  return func(*args, **kwargs)
/Users/robbiati/Documents/tac/lib/python3.12/site-packages/autoray/autoray.py:96: RuntimeWarning: overflow encountered in matmul
  return func(*args, **kwargs)
/Users/robbiati/Documents/tac/lib/python3.12/site-packages/autoray/autoray.py:96: RuntimeWarning: invalid value encountered in matmul
  return func(*args, **kwargs)
/Users/robbiati/Documents/tac/lib/python3.12/site-packages/qibo/hamiltonians/hamiltonians.py:545: RuntimeWarning: divide by zero encountered in matmul
  result = reduce(
/Users/robbiati/Documents/tac/lib/python3.12/site-packages/qibo/hamiltonians/hamiltonians.py:545: RuntimeWarning: overflow encountered in matmul
  result = reduce(
/Users/robbiati/Documents/tac/lib/python3.12/site-packages/qibo/hamiltonians/hamiltonians.py:545: RuntimeWarning: invalid value encountered in matmul
  result = reduce(


MPSTAB expectation value: 0.0013864778434448488, time taken: 0.4858 seconds
Qibo expectation value: 0.0013865313109621127, time taken: 22.8590 seconds


/Users/robbiati/Documents/tac/lib/python3.12/site-packages/qibo/backends/numpy.py:796: RuntimeWarning: divide by zero encountered in matmul
  hstate = hamiltonian @ state
/Users/robbiati/Documents/tac/lib/python3.12/site-packages/qibo/backends/numpy.py:796: RuntimeWarning: overflow encountered in matmul
  hstate = hamiltonian @ state
/Users/robbiati/Documents/tac/lib/python3.12/site-packages/qibo/backends/numpy.py:796: RuntimeWarning: invalid value encountered in matmul
  hstate = hamiltonian @ state


### Supporting generic observables

We leverage Qibo's Symbolic Hamiltonians. 

In [9]:
from qibo.symbols import X, Z
from qibo.hamiltonians import SymbolicHamiltonian

In [13]:
# Defining a pretty random Hamiltonian
nqubits = 9
ham_form = 0
for i in range(8):
    ham_form += (1j) ** i * (i + 1) * Z(i) * Z(i+1) + (i + 1) * 0.3 * X(i) 
h = SymbolicHamiltonian(ham_form)


In [14]:
print(h)

0.3*X0 + 0.6*X1 + 0.9*X2 + 1.2*X3 + 1.5*X4 + 1.8*X5 + 2.1*X6 + 2.4*X7 + 1.0*Z0*Z1 + 2.0*I*Z1*Z2 - 3.0*Z2*Z3 - 4.0*I*Z3*Z4 + 5.0*Z4*Z5 + 6.0*I*Z5*Z6 - 7.0*Z6*Z7 - 8.0*I*Z7*Z8


In [15]:
ans = HardwareEfficient(nqubits=nqubits, nlayers=2)
hs = HSMPO(ansatz=ans)

In [16]:
expval = hs.expectation(observable=h)
expval

/Users/robbiati/Documents/tac/lib/python3.12/site-packages/autoray/autoray.py:96: RuntimeWarning: divide by zero encountered in matmul
  return func(*args, **kwargs)
/Users/robbiati/Documents/tac/lib/python3.12/site-packages/autoray/autoray.py:96: RuntimeWarning: overflow encountered in matmul
  return func(*args, **kwargs)
/Users/robbiati/Documents/tac/lib/python3.12/site-packages/autoray/autoray.py:96: RuntimeWarning: invalid value encountered in matmul
  return func(*args, **kwargs)


-4.6242075655317425

In [17]:
# double check with Qibo
h.expectation_from_state(ans.circuit().state())

/Users/robbiati/Documents/tac/lib/python3.12/site-packages/qibo/hamiltonians/hamiltonians.py:545: RuntimeWarning: divide by zero encountered in matmul
  result = reduce(
/Users/robbiati/Documents/tac/lib/python3.12/site-packages/qibo/hamiltonians/hamiltonians.py:545: RuntimeWarning: overflow encountered in matmul
  result = reduce(
/Users/robbiati/Documents/tac/lib/python3.12/site-packages/qibo/hamiltonians/hamiltonians.py:545: RuntimeWarning: invalid value encountered in matmul
  result = reduce(
/Users/robbiati/Documents/tac/lib/python3.12/site-packages/qibo/backends/numpy.py:796: RuntimeWarning: divide by zero encountered in matmul
  hstate = hamiltonian @ state
/Users/robbiati/Documents/tac/lib/python3.12/site-packages/qibo/backends/numpy.py:796: RuntimeWarning: overflow encountered in matmul
  hstate = hamiltonian @ state
/Users/robbiati/Documents/tac/lib/python3.12/site-packages/qibo/backends/numpy.py:796: RuntimeWarning: invalid value encountered in matmul
  hstate = hamiltonian

np.float64(-4.62420756553182)